# CCW+Pancake Wrap Torque Analysis

This notebook contains analysis that belongs to [SITCOM-1854 CCW+Pancake Wrap Torque Analysis].  
We collected the data on 2025-02-17 and 2025-02-18.  
It uses TDMS files generated by the Camera Cable Wrap (CCW) Engineering User Interface (EUI) and recorded in the host machine.  
The TDMS is a specific format that requires a special package to allow reading these files.  
Use the cell below to install it locally since Rubin Science Pipelines does not have it installed by default.
See the ticket mentioned above to collect the data used in this notebook.

[SITCOM-1854 CCW+Pancake Wrap Torque Analysis]: https://rubinobs.atlassian.net/browse/SITCOM-1854

In [ ]:
# You will need this package since it does not come with RSP
!pip install --quiet nptdms 

Now, let's import some useful packages.

In [ ]:
import glob
import os
import pandas as pd

from matplotlib import pyplot as plt
from matplotlib.dates import DateFormatter
from nptdms import TdmsFile

And define a function that can quickly convert TDMS files and CSV files, which are easier to read.

In [ ]:
def tdms_to_csv(tdms_path: str, csv_path: str):
    """
    Convert a TDMS file to a CSV file.

    Parameters:
    tdms_path (str): Path to the input TDMS file.
    csv_path (str): Path to save the output CSV file.
    """
    tdms_file = TdmsFile.read(tdms_path)

    data = {}
    max_length = 0

    for group in tdms_file.groups():
        for channel in group.channels():
            data[channel.name] = channel[:]
            max_length = max(max_length, len(channel[:]))

    # Ensure all channels have the same length by padding with NaN
    for key in data:
        if len(data[key]) < max_length:
            data[key] = list(data[key]) + [None] * (max_length - len(data[key]))

    df = pd.DataFrame(data)

    df.to_csv(csv_path, index=False)
    print(f"CSV file saved to {csv_path}")

Finally, let's convert these files before starting the actual analysis.

In [ ]:
data_path = "ROT_CCW_FEB_TESTING/20250218_ROT_CCW_Fullrange_Testing/"
tmds_pattern = "*.tdms"
csv_pattern = "*.csv"

In [ ]:
for input_file in glob.glob(os.path.join(data_path, tmds_pattern)):
    output_file = input_file.replace(".tdms", ".csv")
    tdms_to_csv(input_file, output_file)

## Single File Analysis

Let's explore a bit the content of a speficic file.

In [ ]:
df = pd.read_csv(
    "ROT_CCW_FEB_TESTING/20250218_ROT_CCW_Fullrange_Testing/TelemetryData_2025_02_18_14_40.csv"
)
print(df.columns)

We do not need all this information. We are interested only in the CCW position and the torques. Let's define those and let's rename them for convenience. 

In [ ]:
columns = {
    "psp://192.168.209.11/PXIComm_NSV/CCW Current 1": "torque_1",
    "psp://192.168.209.11/PXIComm_NSV/CCW Current 1 TimeStamp": "torque_1_timestamp",
    "psp://192.168.209.11/PXIComm_NSV/CCW Current 2": "torque_2",
    "psp://192.168.209.11/PXIComm_NSV/CCW Current 2 TimeStamp": "torque_2_timestamp",
    "psp://192.168.209.11/PXIComm_NSV/CCW Angle 1": "angle_1",
    "psp://192.168.209.11/PXIComm_NSV/CCW Angle 1 TimeStamp": "angle_1_timestamp",
    "psp://192.168.209.11/PXIComm_NSV/CCW Angle 1": "angle_2",
    "psp://192.168.209.11/PXIComm_NSV/CCW Angle 2 TimeStamp": "angle_2_timestamp",
    "psp://192.168.209.11/PXIComm_NSV/CCW Angle": "angle",
    "psp://192.168.209.11/PXIComm_NSV/CCW Angle TimeStamp": "angle_timestamp",
}

In [ ]:
df = pd.read_csv(
    "ROT_CCW_FEB_TESTING/20250218_ROT_CCW_Fullrange_Testing/TelemetryData_2025_02_18_14_40.csv"
)
df = df[columns.keys()]
df = df.rename(columns=columns)

In [ ]:
for c in columns.values():
    if "timestamp" in c:
        df[c] = pd.to_datetime(df[c], unit="s", utc=True)

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(num="Single File Analysis")

ax.plot(df["torque_1_timestamp"], df["torque_1"], color="C0", label="Motor 1")
ax.plot(df["torque_2_timestamp"], df["torque_2"], color="C1", label="Motor 2")
ax.grid(":", alpha=0.25)
ax.legend()
ax.set_xlabel("Timestamp [TAI]")
ax.set_ylabel("CCW Torques [%]")

date_form = DateFormatter("%H:%M")
ax.xaxis.set_major_formatter(date_form)

ax2 = ax.twinx()
ax2.plot(df["angle_timestamp"], df["angle"], color="C2", label="CCW Angle")
ax2.set_ylabel("CCW Angle [deg]")
ax2.legend()

fig.autofmt_xdate()
fig.suptitle("CCW Torques Percentage")

plt.show()

## Full Analysis

Now, let's put together all the files in a single dataframe.

In [ ]:
full_df = pd.DataFrame(columns=columns.values())

# Concatenate all the files into a single dataframe
for input_file in glob.glob(os.path.join(data_path, csv_pattern)):
    print(f"Reading {input_file}")
    temp_df = pd.read_csv(input_file)
    temp_df = temp_df[columns.keys()]
    temp_df = temp_df.rename(columns=columns)
    temp_df = temp_df.dropna()

    full_df = pd.concat((full_df, temp_df), ignore_index=True)

# Fix the timestamps format
for c in columns.values():
    if "timestamp" in c:
        full_df[c] = pd.to_datetime(full_df[c], unit="s", utc=True)

# Sort by timestamp
full_df = full_df.sort_values("angle_timestamp")

In [ ]:
%matplotlib inline
fig, ax = plt.subplots(num="Full Analysis")

ax.plot(full_df["torque_1_timestamp"], full_df["torque_1"], color="C0", label="Motor 1")
ax.plot(full_df["torque_2_timestamp"], full_df["torque_2"], color="C1", label="Motor 2")
ax.grid(":", alpha=0.25)
ax.legend()
ax.set_xlabel("Timestamp [TAI]")
ax.set_ylabel("CCW Torques [%]")

date_form = DateFormatter("%H:%M")
ax.xaxis.set_major_formatter(date_form)

ax2 = ax.twinx()
ax2.plot(full_df["angle_timestamp"], full_df["angle"], color="C2", label="CCW Angle")
ax2.set_ylabel("CCW Angle [deg]")
ax2.legend()

fig.autofmt_xdate()
fig.suptitle("CCW Torques Percentage")
plt.savefig("ccw_torques_eui_telemetry_20250218.png")

plt.show()

Here are the minimum and maximum values for each motor.

In [ ]:
print(f"Maximum absolute value for Motor 1: {full_df.torque_1.abs().max():.2f} %")
print(f"Maximum absolute value for Motor 2: {full_df.torque_2.abs().max():.2f} %")